# 04 - Clustering Pipeline B y arquetipos mutuamente excluyentes
**Tesis de Maestría en Ciencia de Datos - Universidad Austral**  
*Autor: Nicolás Palavecino | Tutor: Ezequiel Nuske*

---

## 1. K-means++ con k por reglas (no `K_OPTIMAL = 3` como único camino)

- Barrido de k (YAML `k_range`, p.ej. 2–7).
- Si la silueta máxima **≥ 0.25** → ese k; si no → **mínimo Davies-Bouldin**.
- Tamaño mínimo de clúster **≥ 3**.
- DBSCAN solo como **prefiltro de outliers** opcional, no como partición principal.
- Nombres de arquetipo: reglas mutuamente excluyentes sobre las **6–8 features agregadas / centroides**.
- El **radar usa el mismo espacio** que el ajuste por defecto (no top-2 de ~60 columnas).


In [ ]:
import pandas as pd
import numpy as np
from sklearn.datasets import make_blobs

from f1_style_clustering.models.evaluation import select_k_rule_based
from f1_style_clustering.models.clustering import fit_kmeans
from f1_style_clustering.analysis.archetypes import (
    characterize_clusters,
    assign_archetype_names,
    radar_feature_list,
)

X, _ = make_blobs(n_samples=18, centers=3, cluster_std=0.25, random_state=0)
df = pd.DataFrame(X, columns=['brake_point_rel', 'apex_speed_rel'])
# Completar el espacio Pipeline B sintético
for col in ['trail_brake_frac', 'throttle_attack', 'coast_frac', 'full_throttle_frac']:
    df[col] = np.random.default_rng(0).normal(size=len(df))

k_star, eval_df, rule = select_k_rule_based(df, k_min=2, k_max=5, silhouette_threshold=0.25, min_cluster_size=3)
print('k* =', k_star, 'rule =', rule)
display(eval_df)

labels, _ = fit_kmeans(df, n_clusters=k_star)
profiles = characterize_clusters(df, labels)
names = assign_archetype_names(profiles)
print(names)
print('Radar features:', radar_feature_list(df))
assert len(set(names.values())) == len(names)


## 2. Ejecución completa con FastF1 (cache local, no CI)

```python
from f1_style_clustering.pipeline import run_pipeline_b
out = run_pipeline_b(session)  # k por reglas
print(out['k_used'], out['k_rule'], out['archetype_names'])
```

Override explícito: `run_pipeline_b(session, k_clusters=3)` o `--k-clusters 3` en el CLI.
